# 3.2 — Generalization & the IID Assumption

Generalization is the promise that a rule learned from today’s examples will still behave on tomorrow’s examples. In this lesson, we build that promise from empirical risk, the i.i.d. sampling contract, validation checks, complexity costs, score gaps, and stabilizing constraints — always asking whether a prettier training number is actually a durable future decision.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build generalization one idea at a time. Run each cell in order and read the printed intermediate values — every score is just an average, penalty, comparison, or stability adjustment you can inspect. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, random sampling, means, and small numerical checks.
import matplotlib.pyplot as plt  # compact visualizations for risk and validation behavior.
np.random.seed(0)  # reproducibility for simulations and resampling.

### 1. Empirical risk: the training number is an average

A model is judged by a loss on each example, then those losses are averaged. The true future risk is $R(f)=\mathbb E[\ell(f(X),Y)]$, but the quantity we can compute from a dataset is empirical risk $R_S(f)=\frac1m\sum_i \ell(f(x_i),y_i)$. The averaging step matters: one bad example should count, but it should not be confused with the whole distribution.

In [ ]:
losses_w = np.array([0.202, 0.135, 0.539])  # three verified per-example losses from the lesson text.
risk_w = float(np.mean(losses_w))  # empirical risk is the arithmetic mean over examples.
print("losses:", losses_w)  # inspect the individual contributions before averaging.
print("empirical risk:", round(risk_w, 3))  # (0.202 + 0.135 + 0.539) / 3 = 0.292.
assert round(risk_w, 3) == 0.292  # the canonical lesson value.

▶ What you'll see: three small losses become the single train-set score 0.292.

In [ ]:
plt.figure(figsize=(4.4, 3))  # one compact diagnostic plot.
plt.bar(["ex1", "ex2", "ex3"], losses_w, color="steelblue")  # show each example's loss.
plt.axhline(risk_w, color="crimson", linestyle="--", label="mean risk")  # mark the average.
plt.title("1: empirical risk is an average")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: the dashed line sits between the easy examples and the harder example, because the risk summarizes all of them.

*Why it's done this way:* The expectation $R(f)$ is the population average we wish we knew. The sample average $R_S(f)$ is the unbiased plug-in estimate when examples are drawn from the same distribution, so every later model-selection step starts from this mean rather than from the best or worst individual case.

### 2. IID: the sampling contract that makes the average meaningful

The i.i.d. assumption says each training pair is an **independent** draw from the **same** distribution as future pairs. “Independent” keeps one example from secretly duplicating another; “identically distributed” says the train and future mechanisms match. If either part breaks, the training average may no longer point at the future average.

In [ ]:
rng_w = np.random.default_rng(1)  # local random generator for a reproducible sampling demo.
train_iid_w = rng_w.normal(loc=2.0, scale=1.0, size=80)  # training examples from one mechanism.
future_iid_w = rng_w.normal(loc=2.0, scale=1.0, size=80)  # future examples from the same mechanism.
train_shift_w = rng_w.normal(loc=2.0, scale=1.0, size=80)  # training from the old mechanism.
future_shift_w = rng_w.normal(loc=3.0, scale=1.0, size=80)  # future from a shifted mechanism.
print("iid means:", round(float(train_iid_w.mean()), 3), round(float(future_iid_w.mean()), 3))
print("shifted means:", round(float(train_shift_w.mean()), 3), round(float(future_shift_w.mean()), 3))

▶ What you'll see: the iid train/future means are close, while the shifted future mean is about one unit higher.

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(train_iid_w, bins=14, alpha=0.55, label="train iid", color="teal")
plt.hist(future_iid_w, bins=14, alpha=0.55, label="future iid", color="orange")
plt.title("2: same mechanism under IID")
plt.xlabel("x")
plt.ylabel("count")
plt.legend()
plt.show()

▶ What you'll see: the two histograms overlap because they came from the same distribution.

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(train_shift_w, bins=14, alpha=0.55, label="train", color="teal")
plt.hist(future_shift_w, bins=14, alpha=0.55, label="future shifted", color="crimson")
plt.title("2: distribution shift breaks the contract")
plt.xlabel("x")
plt.ylabel("count")
plt.legend()
plt.show()

▶ What you'll see: the shifted future histogram moves right, so a rule tuned to the old train distribution is no longer being tested on the same problem.

*Why it's done this way:* Generalization theory connects $R_S$ to $R$ by treating the sample average as evidence about the same random process that will produce future data. A distribution shift changes the target expectation itself, so the model can have a valid training score and still fail the future task.

### 3. Training fit versus future behavior

A flexible rule can reduce training loss by chasing quirks of the sample. To see that, we fit two polynomial models from scratch with NumPy: a simple line and a high-degree polynomial. The high-degree model has more freedom, so it can lower training error, but validation checks whether the extra freedom survived contact with new i.i.d. data.

In [ ]:
rng_w = np.random.default_rng(2)  # reproducible train/validation samples.
x_train_w = np.linspace(-1, 1, 12)  # small training set.
y_train_w = 1.0 + 2.0 * x_train_w + rng_w.normal(0, 0.25, size=x_train_w.size)  # noisy linear mechanism.
x_val_w = np.linspace(-0.95, 0.95, 80)  # dense validation inputs from the same range.
y_val_w = 1.0 + 2.0 * x_val_w + rng_w.normal(0, 0.25, size=x_val_w.size)  # fresh iid validation labels.
print("train size:", x_train_w.size, "validation size:", x_val_w.size)

▶ What you'll see: a small training sample and a larger validation sample drawn from the same data-generating rule.

In [ ]:
coef_line_w = np.polyfit(x_train_w, y_train_w, deg=1)  # low-complexity hypothesis.
coef_flex_w = np.polyfit(x_train_w, y_train_w, deg=10)  # high-complexity hypothesis.
train_line_w = np.polyval(coef_line_w, x_train_w)  # line predictions on training inputs.
train_flex_w = np.polyval(coef_flex_w, x_train_w)  # flexible predictions on training inputs.
val_line_w = np.polyval(coef_line_w, x_val_w)  # line predictions on validation inputs.
val_flex_w = np.polyval(coef_flex_w, x_val_w)  # flexible predictions on validation inputs.
print("train MSE line/flex:", round(float(np.mean((y_train_w - train_line_w) ** 2)), 3), round(float(np.mean((y_train_w - train_flex_w) ** 2)), 3))
print("val MSE line/flex:", round(float(np.mean((y_val_w - val_line_w) ** 2)), 3), round(float(np.mean((y_val_w - val_flex_w) ** 2)), 3))

▶ What you'll see: the flexible model can look excellent on training while validation exposes whether it paid for that flexibility.

In [ ]:
grid_w = np.linspace(-1, 1, 200)  # smooth grid for plotting fitted functions.
plt.figure(figsize=(5, 3.3))
plt.scatter(x_train_w, y_train_w, color="black", label="train data")
plt.plot(grid_w, np.polyval(coef_line_w, grid_w), color="teal", label="degree 1")
plt.plot(grid_w, np.polyval(coef_flex_w, grid_w), color="crimson", label="degree 10")
plt.title("3: flexibility can chase sample noise")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

▶ What you'll see: the high-degree curve bends to pass near the training points, while the line keeps the reusable trend.

*Why it's done this way:* Empirical risk minimization chooses the rule with low sample loss inside a hypothesis family. Validation uses held-out examples to estimate whether the selected rule captured structure from the distribution or idiosyncrasies of the sample.

### 4. Complexity cost: selection uses the full score

The lesson’s arithmetic adds a cost of 0.070 to the empirical risk. That cost can represent regularization, complexity, operational burden, or any method-specific penalty. The important point is that the selection score is not the raw training average alone.

In [ ]:
cost_w = 0.070  # complexity or regularization cost from the lesson text.
score_w = risk_w + cost_w  # full decision score = empirical risk + cost.
print("empirical risk:", round(risk_w, 3))
print("cost:", round(cost_w, 3))
print("decision score:", round(score_w, 3))
assert round(score_w, 3) == 0.362  # 0.292 + 0.070.

▶ What you'll see: the attractive raw score 0.292 becomes a decision score of 0.362 once cost is included.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["raw risk", "cost", "full score"], [risk_w, cost_w, score_w], color=["teal", "orange", "purple"])
plt.title("4: do not drop the cost term")
plt.ylabel("score units")
plt.show()

▶ What you'll see: the full score is the raw fit plus the extra price of using that method.

*Why it's done this way:* A more flexible method can make $R_S$ smaller by spending capacity. The penalty puts fit and flexibility on the same decision scale, so the chosen model is the one with the best tradeoff rather than the flattiest training residuals.

### 5. Score gaps and relative evidence

A competing flexible alternative has decision score 0.406. Lower is better, so the baseline score 0.362 wins by an absolute gap of 0.044. The relative gap, 10.8%, is often the more honest reading because tiny absolute wins can vanish under resampling noise.

In [ ]:
alt_score_w = 0.406  # score for a tempting alternative.
gap_w = alt_score_w - score_w  # positive means the alternative is worse because lower is better.
relative_gap_w = gap_w / alt_score_w  # scale the gap by the alternative score.
print("baseline score:", round(score_w, 3), "alternative score:", round(alt_score_w, 3))
print("absolute gap:", round(gap_w, 3))
print("relative gap:", round(relative_gap_w, 3))
assert round(gap_w, 3) == 0.044
assert round(relative_gap_w, 3) == 0.108

▶ What you'll see: the baseline wins, but the size of the win is part of the decision evidence.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["baseline", "alternative"], [score_w, alt_score_w], color=["seagreen", "indianred"])
plt.ylabel("decision score (lower better)")
plt.title("5: compare full scores, then read the gap")
plt.show()

▶ What you'll see: two close bars; the lower bar wins, but not by a huge margin.

*Why it's done this way:* Model selection is not just ranking numbers. The gap estimates how much evidence separates options, and the relative gap asks whether that evidence is large on the scale of the scores being compared.

### 6. Stabilization: a constraint can improve the future-facing score

A stabilizing knob that reduces the decision score by 20% gives $0.80\cdot0.362=0.290$. This is the recurring bargain in Part 3: giving up brittle variation can produce a better future-facing rule, even if the raw training fit looked less glamorous.

In [ ]:
stable_score_w = 0.80 * score_w  # apply a 20% stabilizing reduction.
all_scores_w = np.array([score_w, alt_score_w, stable_score_w])  # compare all candidates.
labels_w = np.array(["baseline", "flexible", "stabilized"])  # readable names.
best_idx_w = int(np.argmin(all_scores_w))  # lower score wins.
print("stable score:", round(stable_score_w, 3))
print("winner:", labels_w[best_idx_w], round(float(all_scores_w[best_idx_w]), 3))
assert round(stable_score_w, 3) == 0.290
assert labels_w[best_idx_w] == "stabilized"

▶ What you'll see: the stabilized score is the lowest of the three options.

In [ ]:
plt.figure(figsize=(5, 3))
colors_w = ["gray", "indianred", "seagreen"]
plt.bar(labels_w, all_scores_w, color=colors_w)
plt.ylabel("decision score (lower better)")
plt.title("6: final score comparison")
plt.show()

▶ What you'll see: the stabilized bar is lowest, so it is the model to carry forward in this toy decision.

*Why it's done this way:* Stability is useful because the future sample is not the training sample. A constraint, regularizer, or validation-selected setting can reduce sensitivity to accidental sample details, making the full decision score more trustworthy.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, means, polynomial fits, simulations, and assertions.
import matplotlib.pyplot as plt # load Matplotlib for the small diagnostic plots in this lesson.
np.random.seed(0) # make the notebook's random examples reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Average three losses into empirical risk

**Goal.** Compute the lesson’s empirical risk from three per-example losses, because generalization starts with a sample average. We build it in 2 steps.

In [ ]:
losses_b1 = np.array([0.202, 0.135, 0.539]) # store the verified example losses.
print("losses_b1:", losses_b1) # inspect each observed loss before summarizing.

▶ What you'll see: three separate loss values, not yet a model score.

In [ ]:
risk_b1 = float(np.mean(losses_b1)) # average losses to estimate empirical risk R_S.
print("R_S:", round(risk_b1, 3)) # inspect the train-set risk.
assert round(risk_b1, 3) == 0.292 # verify the lesson value.
plt.figure(figsize=(4, 3)) # create a compact loss plot.
plt.bar(["1", "2", "3"], losses_b1, color="teal") # show example-level losses.
plt.axhline(risk_b1, color="red", linestyle="--") # show their mean.
plt.title("Basic 1: empirical risk") # title the plot.
plt.ylabel("loss") # label the score axis.
plt.show() # display the figure.

▶ What you'll see: the dashed average line summarizes all three losses.

👀 Takeaway: empirical risk is the sample mean of per-example losses.

### Basic 2 — Add the method cost

**Goal.** Add a complexity cost to the raw training risk, because selection should use the full decision score. We build it in 2 steps.

In [ ]:
risk_b2 = 0.292 # reuse the verified empirical risk.
cost_b2 = 0.070 # store the method cost from the lesson.
print("risk:", risk_b2, "cost:", cost_b2) # inspect the two score components.

▶ What you'll see: the raw fit and the extra cost are separate quantities.

In [ ]:
score_b2 = risk_b2 + cost_b2 # combine fit and cost on the same scale.
print("full score:", round(score_b2, 3)) # inspect the selection score.
assert round(score_b2, 3) == 0.362 # verify 0.292 + 0.070.
plt.figure(figsize=(4, 3)) # create a component chart.
plt.bar(["risk", "cost", "score"], [risk_b2, cost_b2, score_b2], color=["teal", "orange", "purple"]) # compare terms.
plt.title("Basic 2: risk plus cost") # title the plot.
plt.ylabel("score units") # label the score scale.
plt.show() # display the chart.

▶ What you'll see: the decision score is larger than raw risk because it includes cost.

👀 Takeaway: optimizing raw training loss alone can choose an unnecessarily flexible method.

### Basic 3 — Compare two decision scores

**Goal.** Rank a baseline and a flexible alternative, because lower full score means the better decision under this objective. We build it in 2 steps.

In [ ]:
scores_b3 = np.array([0.362, 0.406]) # store baseline and flexible alternative scores.
labels_b3 = np.array(["baseline", "flexible"]) # name the two candidates.
print("scores:", dict(zip(labels_b3, scores_b3))) # inspect candidates before selecting.

▶ What you'll see: the flexible alternative has the higher score.

In [ ]:
winner_b3 = labels_b3[int(np.argmin(scores_b3))] # choose the smaller score.
print("winner:", winner_b3) # inspect which model wins.
assert winner_b3 == "baseline" # verify lower score wins here.
plt.figure(figsize=(4, 3)) # create a comparison plot.
plt.bar(labels_b3, scores_b3, color=["seagreen", "indianred"]) # visualize scores.
plt.title("Basic 3: lower full score wins") # title the plot.
plt.ylabel("decision score") # label the score axis.
plt.show() # display the plot.

▶ What you'll see: the baseline bar is lower, so it wins.

👀 Takeaway: compare complete scores, not isolated pieces of the calculation.

### Basic 4 — Compute the absolute gap

**Goal.** Measure how much worse the alternative is, because a ranking without a gap hides the strength of evidence. We build it in 2 steps.

In [ ]:
base_b4 = 0.362 # baseline score.
alt_b4 = 0.406 # alternative score.
print("baseline:", base_b4, "alternative:", alt_b4) # inspect the two scores.

▶ What you'll see: two close decision scores.

In [ ]:
gap_b4 = alt_b4 - base_b4 # positive gap means alternative is worse.
print("gap:", round(gap_b4, 3)) # inspect the absolute difference.
assert round(gap_b4, 3) == 0.044 # verify the lesson gap.
plt.figure(figsize=(4, 3)) # create a gap visualization.
plt.bar(["gap"], [gap_b4], color="slateblue") # plot the score difference.
plt.title("Basic 4: absolute score gap") # title the plot.
plt.ylabel("alternative - baseline") # label the gap.
plt.show() # display the figure.

▶ What you'll see: the baseline wins by 0.044 score units.

👀 Takeaway: score gaps tell you how strong the preference is, not just who ranked first.

### Basic 5 — Compute the relative gap

**Goal.** Put the gap on the score scale, because the same absolute gap can be large or small depending on the denominator. We build it in 2 steps.

In [ ]:
gap_b5 = 0.044 # absolute difference between flexible and baseline scores.
alt_b5 = 0.406 # alternative score used as the scale.
print("gap:", gap_b5, "scale:", alt_b5) # inspect numerator and denominator.

▶ What you'll see: the gap is compared against the alternative’s total score.

In [ ]:
relative_gap_b5 = gap_b5 / alt_b5 # compute the scaled evidence gap.
print("relative gap:", round(relative_gap_b5, 3)) # inspect the fractional difference.
assert round(relative_gap_b5, 3) == 0.108 # verify the lesson value.
plt.figure(figsize=(4, 3)) # create a one-bar plot.
plt.bar(["relative gap"], [relative_gap_b5], color="darkorange") # visualize the fraction.
plt.ylim(0, 0.2) # focus on the relevant scale.
plt.title("Basic 5: gap as a fraction") # title the plot.
plt.show() # display the chart.

▶ What you'll see: the relative gap is about 10.8% of the alternative score.

👀 Takeaway: relative gaps make score differences easier to interpret across scales.

### Basic 6 — Apply a stabilizing reduction

**Goal.** Compute the stabilized score, because constraints can improve the future-facing decision score. We build it in 2 steps.

In [ ]:
score_b6 = 0.362 # baseline full score.
stability_factor_b6 = 0.80 # a 20% reduction leaves 80% of the score.
print("score:", score_b6, "factor:", stability_factor_b6) # inspect the ingredients.

▶ What you'll see: the stabilizing knob is represented as a multiplicative factor.

In [ ]:
stable_b6 = stability_factor_b6 * score_b6 # apply the stabilizing reduction.
print("stable score:", round(stable_b6, 3)) # inspect the stabilized value.
assert round(stable_b6, 3) == 0.290 # verify 0.80 * 0.362.
plt.figure(figsize=(4, 3)) # create a before-after plot.
plt.bar(["before", "stable"], [score_b6, stable_b6], color=["gray", "seagreen"]) # compare scores.
plt.title("Basic 6: stabilization reduces score") # title the chart.
plt.ylabel("decision score") # label the score scale.
plt.show() # display the plot.

▶ What you'll see: the stabilized score is lower than the original full score.

👀 Takeaway: stability is valuable when it reduces the score that represents future performance.

### Basic 7 — Choose the minimum among three scores

**Goal.** Select among baseline, flexible, and stabilized options, because model choice is an end-to-end comparison. We build it in 2 steps.

In [ ]:
scores_b7 = np.array([0.362, 0.406, 0.290]) # store all final candidate scores.
labels_b7 = np.array(["baseline", "flexible", "stabilized"]) # name each candidate.
print("candidates:", list(zip(labels_b7, scores_b7))) # inspect the decision table.

▶ What you'll see: all three scores are on the same lower-is-better scale.

In [ ]:
best_b7 = int(np.argmin(scores_b7)) # locate the smallest score.
print("best:", labels_b7[best_b7], round(float(scores_b7[best_b7]), 3)) # inspect the winner.
assert labels_b7[best_b7] == "stabilized" # verify the toy lesson decision.
plt.figure(figsize=(4.6, 3)) # create the final comparison plot.
plt.bar(labels_b7, scores_b7, color=["gray", "indianred", "seagreen"]) # draw all options.
plt.title("Basic 7: final minimum") # title the plot.
plt.ylabel("decision score") # label the score axis.
plt.show() # display the chart.

▶ What you'll see: the stabilized bar is the lowest.

👀 Takeaway: the correct decision unit is the complete score implied by the method.

### Basic 8 — Simulate train and future means under IID

**Goal.** Compare two independent samples from the same distribution, because iid makes the train average informative about future data. We build it in 2 steps.

In [ ]:
rng_b8 = np.random.default_rng(8) # create reproducible random samples.
train_b8 = rng_b8.normal(0.0, 1.0, size=60) # draw training data from one mechanism.
future_b8 = rng_b8.normal(0.0, 1.0, size=60) # draw future data from the same mechanism.
print("means:", round(float(train_b8.mean()), 3), round(float(future_b8.mean()), 3)) # inspect sample averages.

▶ What you'll see: the two means differ, but not systematically in one direction.

In [ ]:
plt.figure(figsize=(4.6, 3)) # create a compact histogram.
plt.hist(train_b8, bins=12, alpha=0.6, label="train", color="teal") # show train sample.
plt.hist(future_b8, bins=12, alpha=0.6, label="future", color="orange") # show future sample.
plt.title("Basic 8: iid samples overlap") # title the plot.
plt.legend() # show labels.
plt.show() # display the histogram.

▶ What you'll see: the histograms overlap because both samples came from the same distribution.

👀 Takeaway: iid does not mean identical samples; it means identical sampling mechanism.

### Basic 9 — Show a distribution shift

**Goal.** Contrast iid with a shifted future distribution, because training risk can mislead when the mechanism changes. We build it in 2 steps.

In [ ]:
rng_b9 = np.random.default_rng(9) # create reproducible random samples.
train_b9 = rng_b9.normal(0.0, 1.0, size=80) # old training mechanism.
future_b9 = rng_b9.normal(1.0, 1.0, size=80) # shifted future mechanism.
print("mean shift:", round(float(future_b9.mean() - train_b9.mean()), 3)) # inspect the shift.

▶ What you'll see: the future sample has a noticeably larger mean.

In [ ]:
plt.figure(figsize=(4.6, 3)) # create a comparison histogram.
plt.hist(train_b9, bins=14, alpha=0.6, label="train", color="teal") # plot training values.
plt.hist(future_b9, bins=14, alpha=0.6, label="future shifted", color="crimson") # plot shifted future values.
plt.title("Basic 9: shifted future") # title the plot.
plt.legend() # show labels.
plt.show() # display the figure.

▶ What you'll see: the future distribution moves right relative to training.

👀 Takeaway: distribution shift breaks the same-distribution part of iid.

### Basic 10 — Estimate validation risk by holding out data

**Goal.** Compute a validation mean from held-out losses, because future-facing checks should use examples not optimized directly. We build it in 2 steps.

In [ ]:
train_losses_b10 = np.array([0.18, 0.22, 0.20, 0.16]) # losses used during fitting.
val_losses_b10 = np.array([0.25, 0.31, 0.28, 0.35]) # held-out losses used for checking.
print("train losses:", train_losses_b10) # inspect training losses.
print("validation losses:", val_losses_b10) # inspect validation losses.

▶ What you'll see: validation losses are separate from the losses used to tune the model.

In [ ]:
train_risk_b10 = float(np.mean(train_losses_b10)) # average training losses.
val_risk_b10 = float(np.mean(val_losses_b10)) # average validation losses.
print("train risk:", round(train_risk_b10, 3), "validation risk:", round(val_risk_b10, 3)) # compare estimates.
assert round(train_risk_b10, 3) == 0.190 # verify the train average.
assert round(val_risk_b10, 3) == 0.297 # verify the validation average.
plt.figure(figsize=(4, 3)) # create a risk comparison plot.
plt.bar(["train", "validation"], [train_risk_b10, val_risk_b10], color=["teal", "orange"]) # compare averages.
plt.title("Basic 10: hold-out risk") # title the plot.
plt.ylabel("mean loss") # label the loss scale.
plt.show() # display the chart.

▶ What you'll see: validation risk is higher, warning that training fit was optimistic.

👀 Takeaway: held-out validation estimates how the learned rule behaves away from the optimized sample.

## 🟡 Easy

### Easy 1 — Build a tiny train/validation split

**Goal.** Split a small iid dataset into training and validation portions, because future performance must be estimated on data not used for fitting. We build it in 3 steps.

In [ ]:
rng_e1 = np.random.default_rng(11) # create reproducible data.
x_e1 = np.linspace(-1, 1, 20) # create one-dimensional inputs.
y_e1 = 1.0 + 2.0 * x_e1 + rng_e1.normal(0, 0.15, size=x_e1.size) # generate noisy labels.
print("dataset size:", x_e1.size) # inspect total examples.

▶ What you'll see: a small supervised dataset with 20 examples.

In [ ]:
train_idx_e1 = np.arange(0, 14) # use the first 14 examples for fitting in this toy split.
val_idx_e1 = np.arange(14, 20) # hold out the last 6 examples for validation.
x_train_e1, y_train_e1 = x_e1[train_idx_e1], y_e1[train_idx_e1] # training arrays.
x_val_e1, y_val_e1 = x_e1[val_idx_e1], y_e1[val_idx_e1] # validation arrays.
print("train/val sizes:", x_train_e1.size, x_val_e1.size) # inspect split sizes.

▶ What you'll see: the data is divided into examples for fitting and examples for checking.

In [ ]:
plt.figure(figsize=(5, 3)) # create a split plot.
plt.scatter(x_train_e1, y_train_e1, color="teal", label="train") # draw training points.
plt.scatter(x_val_e1, y_val_e1, color="orange", label="validation") # draw validation points.
plt.title("Easy 1: train/validation split") # title the plot.
plt.xlabel("x") # label input axis.
plt.ylabel("y") # label target axis.
plt.legend() # show split labels.
plt.show() # display the figure.

▶ What you'll see: validation points are held aside rather than used for fitting.

👀 Takeaway: a validation set is a small proxy for future examples under the iid contract.

### Easy 2 — Fit simple and flexible models from scratch

**Goal.** Compare a line and a high-degree polynomial, because lower training risk can come from either real structure or sample-chasing flexibility. We build it in 4 steps.

In [ ]:
rng_e2 = np.random.default_rng(12) # create reproducible samples.
x_train_e2 = np.linspace(-1, 1, 14) # training inputs.
y_train_e2 = 1.0 + 2.0 * x_train_e2 + rng_e2.normal(0, 0.2, size=x_train_e2.size) # noisy training labels.
x_val_e2 = np.linspace(-0.9, 0.9, 50) # validation inputs.
y_val_e2 = 1.0 + 2.0 * x_val_e2 + rng_e2.normal(0, 0.2, size=x_val_e2.size) # fresh validation labels.
print("train size:", x_train_e2.size, "validation size:", x_val_e2.size) # inspect data sizes.

▶ What you'll see: separate train and validation samples from the same underlying linear rule.

In [ ]:
coef_simple_e2 = np.polyfit(x_train_e2, y_train_e2, deg=1) # fit a simple linear hypothesis.
coef_flex_e2 = np.polyfit(x_train_e2, y_train_e2, deg=8) # fit a flexible polynomial hypothesis.
print("simple coefficients:", np.round(coef_simple_e2, 3)) # inspect the line.
print("flex coefficient count:", coef_flex_e2.size) # inspect model capacity.

▶ What you'll see: the flexible model has many more coefficients to tune.

In [ ]:
train_mse_simple_e2 = float(np.mean((y_train_e2 - np.polyval(coef_simple_e2, x_train_e2)) ** 2)) # simple train MSE.
train_mse_flex_e2 = float(np.mean((y_train_e2 - np.polyval(coef_flex_e2, x_train_e2)) ** 2)) # flexible train MSE.
val_mse_simple_e2 = float(np.mean((y_val_e2 - np.polyval(coef_simple_e2, x_val_e2)) ** 2)) # simple validation MSE.
val_mse_flex_e2 = float(np.mean((y_val_e2 - np.polyval(coef_flex_e2, x_val_e2)) ** 2)) # flexible validation MSE.
print("train MSE simple/flex:", round(train_mse_simple_e2, 3), round(train_mse_flex_e2, 3)) # inspect train fit.
print("val MSE simple/flex:", round(val_mse_simple_e2, 3), round(val_mse_flex_e2, 3)) # inspect future proxy.

▶ What you'll see: the flexible model usually improves training fit more than validation fit.

In [ ]:
grid_e2 = np.linspace(-1, 1, 200) # smooth plotting grid.
plt.figure(figsize=(5, 3)) # create a fit plot.
plt.scatter(x_train_e2, y_train_e2, color="black", s=20, label="train") # draw data.
plt.plot(grid_e2, np.polyval(coef_simple_e2, grid_e2), color="teal", label="degree 1") # draw simple fit.
plt.plot(grid_e2, np.polyval(coef_flex_e2, grid_e2), color="crimson", label="degree 8") # draw flexible fit.
plt.title("Easy 2: fit versus flexibility") # title the plot.
plt.legend() # show model labels.
plt.show() # display the figure.

▶ What you'll see: the flexible curve bends more because it has more capacity.

👀 Takeaway: validation risk decides whether extra flexibility is useful or brittle.

### Easy 3 — Add a regularization-style cost to model selection

**Goal.** Penalize larger models before comparing them, because a low raw validation loss may not include every relevant cost. We build it in 3 steps.

In [ ]:
raw_val_e3 = np.array([0.120, 0.105, 0.103, 0.101]) # validation losses for four candidate capacities.
complexity_e3 = np.array([1, 2, 4, 8]) # simple proxy for model complexity.
print("raw validation losses:", raw_val_e3) # inspect raw fit.
print("complexities:", complexity_e3) # inspect costs before scaling.

▶ What you'll see: raw validation loss decreases slightly as complexity grows.

In [ ]:
lam_e3 = 0.006 # cost per unit complexity.
full_score_e3 = raw_val_e3 + lam_e3 * complexity_e3 # add complexity cost to each raw loss.
best_raw_e3 = int(np.argmin(raw_val_e3)) # best by raw loss only.
best_full_e3 = int(np.argmin(full_score_e3)) # best by full score.
print("full scores:", np.round(full_score_e3, 3)) # inspect penalized scores.
print("best raw index:", best_raw_e3, "best full index:", best_full_e3) # compare selection rules.

▶ What you'll see: the full score can prefer a simpler model than raw loss alone.

In [ ]:
plt.figure(figsize=(5, 3)) # create a selection plot.
plt.plot(complexity_e3, raw_val_e3, marker="o", label="raw validation") # plot raw losses.
plt.plot(complexity_e3, full_score_e3, marker="s", label="with cost") # plot penalized scores.
plt.title("Easy 3: complexity changes selection") # title the chart.
plt.xlabel("complexity") # label capacity axis.
plt.ylabel("score") # label score axis.
plt.legend() # show curve labels.
plt.show() # display the plot.

▶ What you'll see: the cost curve rises for overly complex candidates.

👀 Takeaway: model selection should match the full objective, not just the prettiest raw term.

### Easy 4 — Bootstrap the uncertainty of a gap

**Goal.** Resample validation losses to estimate gap variability, because a small score advantage can be sampling noise. We build it in 4 steps.

In [ ]:
rng_e4 = np.random.default_rng(14) # create reproducible resampling.
loss_a_e4 = np.array([0.30, 0.25, 0.33, 0.31, 0.28, 0.27, 0.35, 0.29]) # validation losses for model A.
loss_b_e4 = np.array([0.34, 0.27, 0.36, 0.33, 0.31, 0.30, 0.37, 0.32]) # validation losses for model B.
print("mean A/B:", round(float(loss_a_e4.mean()), 3), round(float(loss_b_e4.mean()), 3)) # inspect average losses.

▶ What you'll see: model A has a lower mean loss than model B.

In [ ]:
observed_gap_e4 = float(loss_b_e4.mean() - loss_a_e4.mean()) # positive gap favors model A.
print("observed gap:", round(observed_gap_e4, 3)) # inspect the raw advantage.
assert abs(observed_gap_e4 - 0.0275) < 1e-12 # verify the hand-checkable gap.

▶ What you'll see: the observed advantage is 0.0275 loss units.

In [ ]:
boot_gaps_e4 = [] # store bootstrap gap estimates.
for _e4 in range(500): # run a small bootstrap loop.
    idx_e4 = rng_e4.integers(0, loss_a_e4.size, size=loss_a_e4.size) # resample paired validation cases.
    boot_gaps_e4.append(float(loss_b_e4[idx_e4].mean() - loss_a_e4[idx_e4].mean())) # compute resampled gap.
boot_gaps_e4 = np.array(boot_gaps_e4) # convert to an array for summaries.
print("bootstrap 5/95%:", np.round(np.percentile(boot_gaps_e4, [5, 95]), 3)) # inspect uncertainty interval.

▶ What you'll see: the resampled gap has a range, not just one fixed value.

In [ ]:
plt.figure(figsize=(5, 3)) # create a bootstrap histogram.
plt.hist(boot_gaps_e4, bins=24, color="slateblue", alpha=0.8) # show gap distribution.
plt.axvline(0, color="black", linestyle="--") # mark no difference.
plt.axvline(observed_gap_e4, color="crimson", label="observed") # mark observed gap.
plt.title("Easy 4: validation gap uncertainty") # title the plot.
plt.xlabel("B loss - A loss") # label gap axis.
plt.legend() # show observed marker.
plt.show() # display the histogram.

▶ What you'll see: if the histogram is mostly above zero, A’s advantage is more stable.

👀 Takeaway: gap size should be read with uncertainty, especially on small validation sets.

### Easy 5 — Detect a non-IID validation shift

**Goal.** Compare feature summaries for train and validation data, because a shifted validation set warns that the iid contract may be broken. We build it in 3 steps.

In [ ]:
rng_e5 = np.random.default_rng(15) # create reproducible samples.
train_x_e5 = rng_e5.normal(0.0, 1.0, size=100) # training feature distribution.
val_x_e5 = rng_e5.normal(0.7, 1.0, size=100) # validation distribution with a mean shift.
print("train mean/std:", round(float(train_x_e5.mean()), 3), round(float(train_x_e5.std()), 3)) # inspect train summary.
print("val mean/std:", round(float(val_x_e5.mean()), 3), round(float(val_x_e5.std()), 3)) # inspect validation summary.

▶ What you'll see: the validation mean is noticeably larger than the training mean.

In [ ]:
mean_gap_e5 = float(abs(val_x_e5.mean() - train_x_e5.mean())) # compute mean difference.
pooled_std_e5 = float(np.sqrt((train_x_e5.var() + val_x_e5.var()) / 2)) # compute pooled scale.
standardized_gap_e5 = mean_gap_e5 / pooled_std_e5 # scale shift by typical variation.
print("standardized mean gap:", round(standardized_gap_e5, 3)) # inspect shift strength.
assert standardized_gap_e5 > 0.3 # verify the shift is visible in this toy sample.

▶ What you'll see: the standardized gap is large enough to flag a possible distribution change.

In [ ]:
plt.figure(figsize=(5, 3)) # create a distribution comparison.
plt.hist(train_x_e5, bins=18, alpha=0.6, label="train", color="teal") # plot train features.
plt.hist(val_x_e5, bins=18, alpha=0.6, label="validation", color="orange") # plot validation features.
plt.title("Easy 5: validation distribution shift") # title the chart.
plt.legend() # show sample labels.
plt.show() # display the histogram.

▶ What you'll see: the validation histogram is shifted right relative to training.

👀 Takeaway: validation is most meaningful when it reflects the same mechanism as future data.

## 🔴 Advanced

### Advanced 1 — Sweep model capacity and plot generalization gap

**Goal.** Train polynomial models of increasing degree, because capacity can reduce training loss while increasing the train-validation gap. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(21) # create reproducible data.
x_train_a1 = np.linspace(-1, 1, 16) # small training inputs.
y_train_a1 = 0.5 + 1.5 * x_train_a1 + rng_a1.normal(0, 0.18, size=x_train_a1.size) # noisy linear labels.
x_val_a1 = np.linspace(-0.95, 0.95, 80) # validation inputs.
y_val_a1 = 0.5 + 1.5 * x_val_a1 + rng_a1.normal(0, 0.18, size=x_val_a1.size) # fresh validation labels.
degrees_a1 = np.array([1, 2, 4, 8, 12]) # candidate capacities.
print("degrees:", degrees_a1) # inspect the sweep.

▶ What you'll see: several model capacities will be compared on the same data.

In [ ]:
train_mse_a1 = [] # store train losses.
val_mse_a1 = [] # store validation losses.
for degree_a1 in degrees_a1: # fit one polynomial per degree.
    coef_a1 = np.polyfit(x_train_a1, y_train_a1, deg=int(degree_a1)) # fit from scratch with NumPy.
    train_mse_a1.append(float(np.mean((y_train_a1 - np.polyval(coef_a1, x_train_a1)) ** 2))) # record train MSE.
    val_mse_a1.append(float(np.mean((y_val_a1 - np.polyval(coef_a1, x_val_a1)) ** 2))) # record validation MSE.
train_mse_a1 = np.array(train_mse_a1) # convert to arrays for plotting.
val_mse_a1 = np.array(val_mse_a1) # convert to arrays for plotting.
print("train MSE:", np.round(train_mse_a1, 3)) # inspect training curve.
print("val MSE:", np.round(val_mse_a1, 3)) # inspect validation curve.

▶ What you'll see: training MSE tends to shrink as degree increases.

In [ ]:
gap_a1 = val_mse_a1 - train_mse_a1 # compute generalization gap by degree.
best_degree_a1 = int(degrees_a1[int(np.argmin(val_mse_a1))]) # choose degree by validation.
print("generalization gaps:", np.round(gap_a1, 3)) # inspect optimism of training risk.
print("best validation degree:", best_degree_a1) # inspect selected capacity.

▶ What you'll see: large gaps indicate that training risk is too optimistic.

In [ ]:
plt.figure(figsize=(5, 3)) # create a capacity sweep plot.
plt.plot(degrees_a1, train_mse_a1, marker="o", label="train") # plot train error.
plt.plot(degrees_a1, val_mse_a1, marker="s", label="validation") # plot validation error.
plt.title("Advanced 1: capacity and generalization gap") # title the chart.
plt.xlabel("polynomial degree") # label capacity axis.
plt.ylabel("MSE") # label error axis.
plt.legend() # show lines.
plt.show() # display the plot.

▶ What you'll see: the validation curve is the one used for future-facing model choice.

👀 Takeaway: the generalization gap is the distance between optimized training performance and held-out performance.

### Advanced 2 — Cross-validation from scratch

**Goal.** Average validation loss across folds, because one hold-out split can be noisy. We build it in 4 steps.

In [ ]:
rng_a2 = np.random.default_rng(22) # create reproducible data.
x_a2 = np.linspace(-1, 1, 30) # inputs.
y_a2 = 1.0 - 0.5 * x_a2 + 0.8 * x_a2 ** 2 + rng_a2.normal(0, 0.15, size=x_a2.size) # mildly curved labels.
degrees_a2 = np.array([1, 2, 5]) # candidate polynomial degrees.
fold_ids_a2 = np.arange(x_a2.size) % 5 # deterministic five-fold assignment.
print("fold counts:", np.bincount(fold_ids_a2)) # inspect balanced folds.

▶ What you'll see: five folds with equal numbers of examples.

In [ ]:
cv_scores_a2 = [] # store mean validation loss per degree.
for degree_a2 in degrees_a2: # evaluate each capacity.
    fold_scores_a2 = [] # collect fold losses for this degree.
    for fold_a2 in range(5): # loop over validation folds.
        val_mask_a2 = fold_ids_a2 == fold_a2 # choose one fold as validation.
        train_mask_a2 = ~val_mask_a2 # use remaining folds for training.
        coef_a2 = np.polyfit(x_a2[train_mask_a2], y_a2[train_mask_a2], deg=int(degree_a2)) # fit on training folds.
        pred_a2 = np.polyval(coef_a2, x_a2[val_mask_a2]) # predict validation fold.
        fold_scores_a2.append(float(np.mean((y_a2[val_mask_a2] - pred_a2) ** 2))) # record fold MSE.
    cv_scores_a2.append(float(np.mean(fold_scores_a2))) # average validation losses.
cv_scores_a2 = np.array(cv_scores_a2) # convert to array for selection.
print("CV scores:", np.round(cv_scores_a2, 3)) # inspect cross-validated losses.

▶ What you'll see: each degree receives an average validation score across five folds.

In [ ]:
best_degree_a2 = int(degrees_a2[int(np.argmin(cv_scores_a2))]) # select the lowest cross-validation score.
print("best degree by CV:", best_degree_a2) # inspect the selected degree.
assert best_degree_a2 in degrees_a2 # verify selection came from the candidate grid.

▶ What you'll see: the chosen degree is one of the predefined candidates.

In [ ]:
plt.figure(figsize=(5, 3)) # create a CV score plot.
plt.plot(degrees_a2, cv_scores_a2, marker="o", color="purple") # plot mean fold loss by degree.
plt.axvline(best_degree_a2, color="crimson", linestyle="--", label="best") # mark selected degree.
plt.title("Advanced 2: cross-validation score") # title the chart.
plt.xlabel("degree") # label model capacity.
plt.ylabel("mean fold MSE") # label validation loss.
plt.legend() # show best marker.
plt.show() # display the plot.

▶ What you'll see: cross-validation smooths the decision across multiple validation splits.

👀 Takeaway: cross-validation reduces dependence on a single lucky or unlucky hold-out split.

### Advanced 3 — Regularized linear regression by closed form

**Goal.** Fit ridge-style linear models with different penalties, because regularization shrinks parameters to trade fit for stability. We build it in 5 steps.

In [ ]:
rng_a3 = np.random.default_rng(23) # create reproducible data.
x_train_a3 = np.linspace(-1, 1, 18) # training inputs.
y_train_a3 = 1.0 + 2.0 * x_train_a3 + rng_a3.normal(0, 0.35, size=x_train_a3.size) # noisy labels.
x_val_a3 = np.linspace(-0.9, 0.9, 60) # validation inputs.
y_val_a3 = 1.0 + 2.0 * x_val_a3 + rng_a3.normal(0, 0.35, size=x_val_a3.size) # fresh validation labels.
print("train/val:", x_train_a3.size, x_val_a3.size) # inspect data sizes.

▶ What you'll see: a noisy line-fitting problem with separate validation data.

In [ ]:
X_train_a3 = np.column_stack([np.ones_like(x_train_a3), x_train_a3]) # design matrix with intercept and slope.
X_val_a3 = np.column_stack([np.ones_like(x_val_a3), x_val_a3]) # validation design matrix.
lams_a3 = np.array([0.0, 0.1, 1.0, 10.0]) # candidate regularization strengths.
print("lambda grid:", lams_a3) # inspect penalty values.

▶ What you'll see: penalties range from none to strong shrinkage.

In [ ]:
val_mse_a3 = [] # store validation losses.
coef_norm_a3 = [] # store parameter sizes.
for lam_a3 in lams_a3: # fit one ridge model per lambda.
    penalty_a3 = lam_a3 * np.eye(X_train_a3.shape[1]) # ridge penalty matrix.
    penalty_a3[0, 0] = 0.0 # do not penalize the intercept in this demo.
    coef_a3 = np.linalg.solve(X_train_a3.T @ X_train_a3 + penalty_a3, X_train_a3.T @ y_train_a3) # closed-form ridge solution.
    pred_val_a3 = X_val_a3 @ coef_a3 # validation predictions.
    val_mse_a3.append(float(np.mean((y_val_a3 - pred_val_a3) ** 2))) # validation MSE.
    coef_norm_a3.append(float(np.linalg.norm(coef_a3[1:]))) # size of penalized coefficients.
print("validation MSE:", np.round(val_mse_a3, 3)) # inspect fit quality.
print("slope norms:", np.round(coef_norm_a3, 3)) # inspect shrinkage.

▶ What you'll see: larger λ shrinks the slope and changes validation error.

In [ ]:
best_lam_a3 = float(lams_a3[int(np.argmin(val_mse_a3))]) # select lambda by validation loss.
print("best lambda:", best_lam_a3) # inspect selected penalty.
assert best_lam_a3 in lams_a3 # verify grid selection.

▶ What you'll see: validation selects the penalty that best balances fit and shrinkage.

In [ ]:
plt.figure(figsize=(5, 3)) # create a regularization plot.
plt.plot(lams_a3, val_mse_a3, marker="o", label="validation MSE") # plot validation loss.
plt.plot(lams_a3, coef_norm_a3, marker="s", label="slope norm") # plot parameter size.
plt.xscale("symlog") # show zero and positive lambdas compactly.
plt.title("Advanced 3: regularization and stability") # title the chart.
plt.xlabel("lambda") # label regularization axis.
plt.legend() # show curves.
plt.show() # display the plot.

▶ What you'll see: regularization reduces parameter size, and validation decides how much shrinkage helps.

👀 Takeaway: regularization is a controlled way to buy stability by limiting parameter magnitude.

### Advanced 4 — Measure dependence from duplicated examples

**Goal.** Show why independence matters, because duplicated or clustered examples make the sample average look more certain than it is. We build it in 4 steps.

In [ ]:
rng_a4 = np.random.default_rng(24) # create reproducible samples.
independent_a4 = rng_a4.normal(0.0, 1.0, size=80) # eighty independent observations.
base_a4 = rng_a4.normal(0.0, 1.0, size=20) # twenty genuinely independent observations.
duplicated_a4 = np.repeat(base_a4, 4) # repeat each one four times to fake a larger dataset.
print("sizes:", independent_a4.size, duplicated_a4.size) # inspect equal apparent sample sizes.

▶ What you'll see: both arrays have 80 entries, but one has only 20 unique draws repeated.

In [ ]:
unique_count_a4 = np.unique(np.round(duplicated_a4, 12)).size # count unique values after rounding.
print("unique duplicated values:", unique_count_a4) # inspect effective variety.
assert unique_count_a4 == 20 # verify duplication reduced independence.

▶ What you'll see: the duplicated dataset has only 20 distinct values.

In [ ]:
means_ind_a4 = [] # store bootstrap means for independent data.
means_dup_a4 = [] # store bootstrap means for duplicated data.
for _a4 in range(400): # run bootstrap resampling.
    idx_ind_a4 = rng_a4.integers(0, independent_a4.size, independent_a4.size) # resample independent entries.
    idx_dup_a4 = rng_a4.integers(0, duplicated_a4.size, duplicated_a4.size) # resample duplicated entries.
    means_ind_a4.append(float(independent_a4[idx_ind_a4].mean())) # bootstrap independent mean.
    means_dup_a4.append(float(duplicated_a4[idx_dup_a4].mean())) # bootstrap duplicated mean.
means_ind_a4 = np.array(means_ind_a4) # convert to arrays.
means_dup_a4 = np.array(means_dup_a4) # convert to arrays.
print("bootstrap std independent/duplicated:", round(float(means_ind_a4.std()), 3), round(float(means_dup_a4.std()), 3)) # inspect uncertainty.

▶ What you'll see: repeated data can give misleading impressions about uncertainty and evidence.

In [ ]:
plt.figure(figsize=(5, 3)) # create uncertainty comparison.
plt.hist(means_ind_a4, bins=22, alpha=0.6, label="independent", color="teal") # plot independent bootstrap means.
plt.hist(means_dup_a4, bins=22, alpha=0.6, label="duplicated", color="orange") # plot duplicated bootstrap means.
plt.title("Advanced 4: dependence changes evidence") # title the plot.
plt.xlabel("bootstrap mean") # label mean axis.
plt.legend() # show labels.
plt.show() # display the histogram.

▶ What you'll see: apparent sample size and effective independent evidence are not the same thing.

👀 Takeaway: independence prevents one repeated observation from masquerading as many examples.

### Advanced 5 — End-to-end model decision with cost, gap, and stability

**Goal.** Combine empirical risk, cost, alternative comparison, and stabilization, because real model choice uses the whole decision pipeline. We build it in 5 steps.

In [ ]:
loss_table_a5 = np.array([[0.202, 0.135, 0.539], [0.180, 0.150, 0.470], [0.220, 0.140, 0.500]]) # losses for baseline, flexible, stabilized candidates.
labels_a5 = np.array(["baseline", "flexible", "stabilized"]) # candidate names.
print("loss table shape:", loss_table_a5.shape) # inspect candidates by examples.

▶ What you'll see: three candidate methods, each with three example losses.

In [ ]:
risk_a5 = loss_table_a5.mean(axis=1) # compute empirical risk per candidate.
cost_a5 = np.array([0.070, 0.139, 0.040]) # attach method costs.
score_a5 = risk_a5 + cost_a5 # compute full decision scores.
print("risks:", np.round(risk_a5, 3)) # inspect raw averages.
print("scores:", np.round(score_a5, 3)) # inspect cost-adjusted scores.
assert round(float(score_a5[0]), 3) == 0.362 # verify baseline lesson score.

▶ What you'll see: raw averages and full scores can rank candidates differently.

In [ ]:
stable_factor_a5 = 0.80 # model the stabilizing knob as a 20% reduction.
score_a5[2] = stable_factor_a5 * score_a5[0] # set the stabilized candidate to the lesson's stable score.
gap_a5 = score_a5[1] - score_a5[0] # compare flexible against baseline.
relative_gap_a5 = gap_a5 / score_a5[1] # scale the gap by flexible score.
print("adjusted scores:", np.round(score_a5, 3)) # inspect the final decision scores.
print("flex-baseline gap:", round(float(gap_a5), 3), "relative:", round(float(relative_gap_a5), 3)) # inspect evidence.
assert round(float(score_a5[2]), 3) == 0.290 # verify stabilized score.

▶ What you'll see: the stabilized candidate inherits the explicit 20% stability improvement.

In [ ]:
best_idx_a5 = int(np.argmin(score_a5)) # choose the smallest final score.
print("winner:", labels_a5[best_idx_a5], round(float(score_a5[best_idx_a5]), 3)) # inspect final selection.
assert labels_a5[best_idx_a5] == "stabilized" # verify the end-to-end toy decision.

▶ What you'll see: stabilized is the selected method after all adjustments.

In [ ]:
plt.figure(figsize=(5, 3)) # create final decision plot.
plt.bar(labels_a5, score_a5, color=["gray", "indianred", "seagreen"]) # plot all full scores.
plt.ylabel("final decision score") # label score axis.
plt.title("Advanced 5: complete decision pipeline") # title the chart.
plt.show() # display the plot.

▶ What you'll see: the final comparison uses full scores, not isolated training fragments.

👀 Takeaway: a generalization-aware decision keeps raw fit, cost, uncertainty, and stability in one pipeline.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Generalization is possible when train and future examples are drawn by the same mechanism.

Generalization & the i.i.d. assumption uses empirical risk, validation behavior, and a cost-aware decision score. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_breast_cancer, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))
    return rungs

def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)

def logistic_baseline(x_tr, y_tr, x_te):
    """Default classifier used to demonstrate a ladder end to end."""
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


## The concept, built once on D1

The lesson formula is $$ R(f)=\mathbb E[\ell(f(X),Y)],\qquad R_S(f)=\frac1m\sum_{i=1}^m\ell(f(x_i),y_i) $$. The next cell recomputes the exact loss average, cost, gap, and stabilized score from the plan.

In [ ]:

def generalization_the_i_i_d_assumption_method():
    losses = np.array([0.202, 0.135, 0.539], dtype=float)
    raw_sum = float(losses.sum())
    empirical_risk = round(float(raw_sum / len(losses)), 3)
    cost = 0.070
    score = round(empirical_risk + cost, 3)
    alternative = 0.406
    gap = round(alternative - score, 3)
    relative_gap = round(gap / alternative, 3)
    stable_score = round(0.80 * score, 3)
    final_score = min(score, alternative, stable_score)
    return {
        "losses": losses,
        "sum": raw_sum,
        "risk": empirical_risk,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
        "relative_gap": relative_gap,
        "stable": stable_score,
        "final": final_score,
    }

lesson_check = generalization_the_i_i_d_assumption_method()
print("losses:", lesson_check["losses"])
print("R_S =", round(lesson_check["sum"], 3), "/ 3 =", round(lesson_check["risk"], 3))
print("score =", round(lesson_check["score"], 3))
print("gap =", round(lesson_check["gap"], 3))
print("relative gap =", round(lesson_check["relative_gap"], 3))
print("stable score =", round(lesson_check["stable"], 3))
assert np.isclose(round(lesson_check["sum"], 3), 0.876)
assert np.isclose(round(lesson_check["risk"], 3), 0.292)
assert np.isclose(round(lesson_check["score"], 3), 0.362)
assert np.isclose(round(lesson_check["gap"], 3), 0.044)
assert np.isclose(round(lesson_check["relative_gap"], 3), 0.108)
assert np.isclose(round(lesson_check["stable"], 3), 0.290)


The assertions above keep the notebook and lesson prose on the same algorithmic scale.

In [ ]:

def safe_stratify(y):
    values, counts = np.unique(y, return_counts=True)
    if len(values) < 2:
        return None
    if counts.min() < 2:
        return None
    return y

def plot_2d_projection(ax, X, y, title):
    x_plot = X[:, :2]
    ax.scatter(x_plot[:, 0], x_plot[:, 1], c=y, cmap="viridis", s=16, alpha=0.75)
    ax.set_title(title, fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

def logistic_candidates_for_rung(X, y):
    stratify = safe_stratify(y)
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
    scaler = StandardScaler()
    x_tr_s = scaler.fit_transform(x_tr)
    x_te_s = scaler.transform(x_te)
    candidates = []
    for c_value in [0.05, 0.2, 1.0, 5.0]:
        model = LogisticRegression(C=c_value, max_iter=2000)
        model.fit(x_tr_s, y_tr)
        tr_prob = model.predict_proba(x_tr_s)
        te_prob = model.predict_proba(x_te_s)
        tr_pred = model.predict(x_tr_s)
        te_pred = model.predict(x_te_s)
        labels = model.classes_
        train_loss = log_loss(y_tr, tr_prob, labels=labels)
        val_loss = log_loss(y_te, te_prob, labels=labels)
        cost = 0.02 / c_value
        candidates.append({
            "C": c_value,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "gap": float(val_loss - train_loss),
            "accuracy": float(accuracy_score(y_te, te_pred)),
            "cost": float(cost),
            "score": float(val_loss + cost),
            "pred": te_pred,
        })
    raw_winner = min(candidates, key=lambda item: item["val_loss"])
    fixed_winner = min(candidates, key=lambda item: item["score"])
    return candidates, raw_winner, fixed_winner

def run_logistic_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        candidates, raw_winner, fixed_winner = logistic_candidates_for_rung(X, y)
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": fixed_winner["val_loss"],
            "gap": fixed_winner["gap"],
            "accuracy": fixed_winner["accuracy"],
            "C": fixed_winner["C"],
            "score": fixed_winner["score"],
            "raw_C": raw_winner["C"],
        })
    return rows

def bias_variance_candidates_for_rung(X, y):
    stratify = safe_stratify(y)
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
    models = [
        ("linear", make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))),
        ("flexible-knn", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))),
    ]
    rows = []
    for label, model in models:
        model.fit(x_tr, y_tr)
        train_error = 1.0 - accuracy_score(y_tr, model.predict(x_tr))
        val_error = 1.0 - accuracy_score(y_te, model.predict(x_te))
        variance_proxy = abs(val_error - train_error)
        complexity_cost = 0.01 if label == "linear" else 0.04
        rows.append({
            "label": label,
            "train_error": float(train_error),
            "val_error": float(val_error),
            "variance_proxy": float(variance_proxy),
            "score": float(val_error + variance_proxy + complexity_cost),
        })
    return rows

def run_bias_variance_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        candidates = bias_variance_candidates_for_rung(X, y)
        winner = min(candidates, key=lambda item: item["score"])
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": winner["val_error"],
            "gap": winner["variance_proxy"],
            "model": winner["label"],
            "score": winner["score"],
        })
    return rows

def add_intercept(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack([ones, X])

def train_binary_perceptron(X, y_signed, epochs=60):
    X_aug = add_intercept(X)
    weights = np.zeros(X_aug.shape[1])
    mistakes = []
    for epoch in range(epochs):
        errors = 0
        for xi, yi in zip(X_aug, y_signed):
            margin = yi * float(np.dot(weights, xi))
            if margin <= 0:
                weights = weights + yi * xi
                errors = errors + 1
        mistakes.append(errors)
        if errors == 0:
            break
    return weights, mistakes

def train_ovr_perceptron(X, y, epochs=60):
    classes = np.unique(y)
    weights = []
    histories = []
    for cls in classes:
        y_signed = np.where(y == cls, 1, -1)
        w, hist = train_binary_perceptron(X, y_signed, epochs=epochs)
        weights.append(w)
        histories.append(hist)
    return classes, np.vstack(weights), histories

def predict_ovr_perceptron(classes, weights, X):
    scores = add_intercept(X).dot(weights.T)
    return classes[np.argmax(scores, axis=1)]

def perceptron_predictor(x_tr, y_tr, x_te):
    classes, weights, histories = train_ovr_perceptron(x_tr, y_tr, epochs=60)
    return predict_ovr_perceptron(classes, weights, x_te)

def run_perceptron_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        stratify = safe_stratify(y)
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
        scaler = StandardScaler()
        x_tr_s = scaler.fit_transform(x_tr)
        x_te_s = scaler.transform(x_te)
        classes, weights, histories = train_ovr_perceptron(x_tr_s, y_tr, epochs=60)
        pred = predict_ovr_perceptron(classes, weights, x_te_s)
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": float(accuracy_score(y_te, pred)),
            "history": histories,
        })
    return rows


## The dataset ladder

D1 is inspectable by hand; D5 is a real 30-dimensional breast-cancer classification problem.

In [ ]:

rungs = clf_ladder()
for name, X, y in rungs:
    values, counts = np.unique(y, return_counts=True)
    preview = np.round(X[:3, :min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  class counts:", dict(zip(values.tolist(), counts.tolist())))
    print("  sample columns:")
    print(preview)


## Run the same method across D1–D5

The metric follows the plan: validation loss and generalization gap for 3.1–3.3, accuracy for 3.4.

In [ ]:

results = run_logistic_ladder()
print("rung | validation_loss | generalization_gap | accuracy | C | score")
for row in results:
    print(f"D{row['rung']} | {row['metric']:.3f} | {row['gap']:.3f} | {row['accuracy']:.3f} | {row['C']} | {row['score']:.3f}")
helper_acc = clf_accuracy(logistic_baseline, rungs[-1][1], rungs[-1][2])
print("D5 logistic_baseline accuracy:", round(helper_acc, 3))


## Results visualization

The closing figure has one panel per rung plus a summary curve over D1–D5.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
flat_axes = axes.ravel()
for ax, (name, X, y), row in zip(flat_axes[:5], rungs, results):
    plot_2d_projection(ax, X, y, f"D{row['rung']} validation loss={row['metric']:.2f}")
flat_axes[5].plot([row["rung"] for row in results], [row["metric"] for row in results], marker="o", label="validation loss")
if "class" != "perceptron":
    flat_axes[5].plot([row["rung"] for row in results], [abs(row["gap"]) for row in results], marker="s", label="gap")
flat_axes[5].set_xlabel("rung")
flat_axes[5].set_ylabel("loss / gap")
flat_axes[5].legend()
fig.tight_layout()
plt.show()


## Pitfall on D5: optimizing the raw term and forgetting the cost

The hardest rung demonstrates why the raw term alone is not the decision rule.

In [ ]:

d5_name, d5_X, d5_y = rungs[-1]
d5_candidates, raw_winner, fixed_winner = logistic_candidates_for_rung(d5_X, d5_y)
print("D5:", d5_name)
print("wrong raw winner C:", raw_winner["C"], "validation loss", round(raw_winner["val_loss"], 3))
print("fixed winner C:", fixed_winner["C"], "validation loss + cost", round(fixed_winner["score"], 3))
print("fixed gap check:", round(fixed_winner["gap"], 3))
assert fixed_winner["score"] <= raw_winner["val_loss"] + raw_winner["cost"] + 1e-9


## Evaluate it + Practice

- Compare the metric with a no-skill baseline or `logistic_baseline`.
- Sanity check: shuffle labels and confirm the score degrades.
- Ablation: turn off the cost or scaling fix and watch the D5 choice or metric change.
- Failure signals: a large validation gap, a scale mismatch, or a raw-only winner.

Practice 1: Change one cost and rerun the D5 selection.

Practice 2: Repeat the D5 split with a different seed and compare the gap.

Practice 3: For skewed classes, add macro-F1 and compare it with accuracy.